# CPI & Unemployment Analysis

**Study Overview:**

This analysis examines how ATM straddles on 15 liquid sector ETFs behave around CPI release dates and Unemployment (Non-Farm Payrolls) report dates from 2016 to 2025. The goal is to understand whether implied volatility around these macro events creates systematic straddle trading opportunities, mirroring the methodology used in the FOMC analysis.

## Part 1: Data Loading & Processing

In [1]:
# Import neccessary Libraries
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

In [2]:
# Load the dataset
df = pd.read_csv("fomc_etf_data/cpi_unemployment_etf_options.csv")
df["event_date"] = pd.to_datetime(df["event_date"])
df["entry_date"] = pd.to_datetime(df["entry_date"])
df["exit_date"]  = pd.to_datetime(df["exit_date"])

In [3]:
# Dedup - keep highest entry_straddle_mid per combo key
KEY = ["ticker","event_date","event_type","entry_offset","exit_offset"]
before = len(df)
df = (df.sort_values("entry_straddle_mid", ascending=False)
        .drop_duplicates(subset=KEY, keep="first")
        .reset_index(drop=True))
print(f"Dedup: {before:,} -> {len(df):,} rows  (dropped {before-len(df):,})")

Dedup: 16,566 -> 16,566 rows  (dropped 0)


In [4]:
# Evaluate Overview
print("Dataset Overview:")
print(f"  Rows: {len(df):,}")
print(f"  Tickers: {df['ticker'].nunique()}  {sorted(df['ticker'].unique())}")
print(f"  Event types: {sorted(df['event_type'].unique())}")
print(f"  CPI events: {df[df['event_type']=='CPI']['event_date'].nunique()}")
print(f"  Unemployment events: {df[df['event_type']=='Unemployment']['event_date'].nunique()}")
print(f"  Date range: {df['event_date'].min().date()} -> {df['event_date'].max().date()}")
print(f"  Years: {sorted(df['year'].unique())}")
print(f"  Entry offsets: {sorted(df['entry_offset'].unique())}")
print(f"  Exit  offsets: {sorted(df['exit_offset'].unique())}")

Dataset Overview:
  Rows: 16,566
  Tickers: 15  ['EEM', 'GLD', 'ITA', 'IWM', 'QQQ', 'SPY', 'TLT', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLV', 'XOP']
  Event types: ['CPI', 'Unemployment']
  CPI events: 126
  Unemployment events: 119
  Date range: 2016-01-08 -> 2025-12-18
  Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Entry offsets: [np.int64(-3), np.int64(-2), np.int64(-1), np.int64(0)]
  Exit  offsets: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


In [5]:
# Rows per ticker per event type with coverage metric
VALID_COMBOS = 15  # 16 combos minus T0/T0

for evt in ["CPI", "Unemployment"]:
    sub = df[df["event_type"] == evt]
    total_events = sub["event_date"].nunique()
    expected = total_events * VALID_COMBOS
    print(f"\n{evt} — {total_events} events, {expected} expected rows per ticker:")
    ticker_summary = (sub.groupby("ticker")
                        .agg(rows=("pnl_pct","count"),
                             avg_ret=("pnl_pct","mean"),
                             non_null=("pnl_pct",lambda x: x.notna().sum()))
                        .assign(coverage=lambda d: d["rows"] / expected * 100)
                        .sort_values("rows", ascending=False))
    print(ticker_summary.to_string(formatters={"avg_ret": "{:.2%}".format, "coverage": "{:.1f}%".format}))


CPI — 126 events, 1890 expected rows per ticker:
        rows avg_ret  non_null coverage
ticker                                 
GLD     1860  -1.28%      1860    98.4%
QQQ     1476   1.66%      1476    78.1%
IWM      937  -0.96%       937    49.6%
SPY      897   3.75%       897    47.5%
XLK      771   4.13%       771    40.8%
XLF      671   0.26%       671    35.5%
XOP      370  -0.90%       370    19.6%
XLE      356  -0.11%       356    18.8%
XLI      227  -6.56%       227    12.0%
EEM      220   2.22%       220    11.6%
XLV      212  -4.92%       212    11.2%
XLB      153  -0.34%       153     8.1%
TLT      146 -11.66%       146     7.7%
ITA      101  -6.81%       101     5.3%
XLP       85 -17.74%        85     4.5%

Unemployment — 119 events, 1785 expected rows per ticker:
        rows avg_ret  non_null coverage
ticker                                 
GLD     1736  -2.12%      1736    97.3%
QQQ     1348  -0.98%      1348    75.5%
SPY      835  -0.92%       835    46.8%
IWM      81

In [6]:
# Check missing data
print("Missing Data:")
key_cols = ["pnl_pct","entry_straddle_mid","exit_straddle_mid",
            "entry_atm_iv","exit_atm_iv","stock_move"]
print(df[key_cols].isnull().sum().to_string())

Missing Data:
pnl_pct                0
entry_straddle_mid     0
exit_straddle_mid      0
entry_atm_iv           0
exit_atm_iv           35
stock_move             0


## Part 2: Unemployment Configuration Analysis

We first analyse straddle returns around Unemployment (Non-Farm Payrolls) report dates, examining how each entry/exit timing combination performs for every ticker.

In [7]:
# Import neccessary libraries
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [8]:
# Set up NUSA standard visualization template
BG    = "#ffffff"
PANEL = "#f8f8f8"
RED   = "#e8133a"
GREEN = "#2db82d"
GREY  = "#555555"
DG    = "#dddddd"
FONT  = "Garamond, 'EB Garamond', Georgia, serif"

In [9]:
# Set e_offs and x_offs for consistent ordering
e_offs = [-3, -2, -1, 0]
x_offs = [0, 1, 2, 3]
df = df[~((df["entry_offset"]==0) & (df["exit_offset"]==0))].copy()

# Split by event type
df_unemp = df[df["event_type"] == "Unemployment"].copy()
df_cpi   = df[df["event_type"] == "CPI"].copy()

In [10]:
# Set up Ticker Configurations
TICKER_META = {
    "SPY": "S&P 500",
    "QQQ": "Nasdaq 100",
    "XLK": "Technology",
    "XLF": "Financials",
    "XLE": "Energy",
    "XLI": "Industrials",
    "XLV": "Health Care",
    "XLB": "Materials",
    "XLP": "Consumer Staples",
    "GLD": "Gold",
    "TLT": "20+ Yr Treasuries",
    "IWM": "Russell 2000",
    "EEM": "Emerging Markets",
    "XOP": "Oil & Gas E&P",
    "ITA": "Aerospace & Defense",
}

In [11]:
# Function to plot heatmap for a given ticker and event subset
def plot_ticker(data, ticker, event_label):
    sub  = data[data["ticker"] == ticker]
    name = TICKER_META.get(ticker, ticker)

    st = (sub.groupby(["entry_offset","exit_offset"])
             .agg(
                 n        = ("pnl_pct","count"),
                 avg_ret  = ("pnl_pct","mean"),
                 win_rate = ("pnl_pct", lambda x: (x>0).mean()),
             )
             .reset_index())
    idx_t = st.set_index(["entry_offset","exit_offset"])

    z, txt = [], []
    for e in e_offs:
        rz, rt = [], []
        for x in x_offs:
            if e == 0 and x == 0:
                rz.append(None); rt.append("excl.")
            elif (e, x) in idx_t.index:
                v  = idx_t.loc[(e,x), "avg_ret"]
                wr = idx_t.loc[(e,x), "win_rate"]
                n  = int(idx_t.loc[(e,x), "n"])
                rz.append(v)
                rt.append(f"{v:.2%}\nWR {wr:.0%}  n={n}")
            else:
                rz.append(None); rt.append("")
        z.append(rz); txt.append(rt)

    fig = go.Figure(go.Heatmap(
        z=z,
        x=[f"Exit T{x:+d}" for x in x_offs],
        y=[f"Entry T{e:+d}" for e in e_offs],
        text=txt, texttemplate="%{text}",
        textfont=dict(family=FONT, size=11, color="#111111"),
        colorscale=[[0,RED],[0.45,"#f5a0aa"],[0.5,"#ffffff"],
                    [0.55,"#a0e6a0"],[1,GREEN]],
        zmid=0, showscale=True,
        colorbar=dict(tickformat=".0%",
                      tickfont=dict(family=FONT, color=GREY, size=11),
                      outlinecolor=DG, outlinewidth=1),
        hoverongaps=False,
    ))
    fig.update_layout(
        title=dict(text=f"{ticker}  ·  {name}  -  {event_label}  -  Avg Return per Combo",
                   font=dict(family=FONT, size=19, color="#111111"),
                   x=0.03, xanchor="left"),
        paper_bgcolor=BG, plot_bgcolor=PANEL,
        font=dict(family=FONT, color="#111111", size=13),
        height=380, margin=dict(l=65, r=40, t=65, b=40),
        hoverlabel=dict(bgcolor="#ffffff", bordercolor=DG,
                        font=dict(family=FONT, color="#111111", size=13)),
    )
    fig.update_xaxes(side="top", tickfont=dict(family=FONT, color="#111111", size=12),
                     gridcolor=DG, linecolor=DG)
    fig.update_yaxes(autorange="reversed",
                     tickfont=dict(family=FONT, color="#111111", size=12),
                     gridcolor=DG, linecolor=DG)
    fig.show()

In [12]:
# Unemployment heatmaps for all tickers
for ticker in TICKER_META:
    plot_ticker(df_unemp, ticker, "Unemployment")

## Part 3: CPI Configuration Analysis

Now we examine the same straddle configurations around CPI release dates.

In [13]:
# CPI heatmaps for all tickers
for ticker in TICKER_META:
    plot_ticker(df_cpi, ticker, "CPI")

## Part 4: Combined CPI + Unemployment Analysis

Here we pool all CPI and Unemployment events together to identify configurations that work across both macro catalysts. This gives larger sample sizes and reveals which straddle setups are robust to the type of macro release.

In [14]:
# Combined heatmaps for all tickers
for ticker in TICKER_META:
    plot_ticker(df, ticker, "CPI + Unemployment (Combined)")

In [15]:
# Side-by-side comparison: CPI vs Unemployment vs Combined for key tickers
def plot_comparison(ticker):
    name = TICKER_META.get(ticker, ticker)
    datasets = [
        (df_unemp, "Unemployment"),
        (df_cpi, "CPI"),
        (df, "Combined"),
    ]

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[d[1] for d in datasets],
        horizontal_spacing=0.08,
    )

    for col_idx, (data, label) in enumerate(datasets, 1):
        sub = data[data["ticker"] == ticker]
        st = (sub.groupby(["entry_offset","exit_offset"])
                 .agg(
                     n        = ("pnl_pct","count"),
                     avg_ret  = ("pnl_pct","mean"),
                     win_rate = ("pnl_pct", lambda x: (x>0).mean()),
                 )
                 .reset_index())
        idx_t = st.set_index(["entry_offset","exit_offset"])

        z, txt = [], []
        for e in e_offs:
            rz, rt = [], []
            for x in x_offs:
                if e == 0 and x == 0:
                    rz.append(None); rt.append("excl.")
                elif (e, x) in idx_t.index:
                    v  = idx_t.loc[(e,x), "avg_ret"]
                    wr = idx_t.loc[(e,x), "win_rate"]
                    n  = int(idx_t.loc[(e,x), "n"])
                    rz.append(v)
                    rt.append(f"{v:.2%}\nWR {wr:.0%}  n={n}")
                else:
                    rz.append(None); rt.append("")
            z.append(rz); txt.append(rt)

        fig.add_trace(
            go.Heatmap(
                z=z,
                x=[f"Exit T{x:+d}" for x in x_offs],
                y=[f"Entry T{e:+d}" for e in e_offs],
                text=txt, texttemplate="%{text}",
                textfont=dict(family=FONT, size=10, color="#111111"),
                colorscale=[[0,RED],[0.45,"#f5a0aa"],[0.5,"#ffffff"],
                            [0.55,"#a0e6a0"],[1,GREEN]],
                zmid=0, showscale=(col_idx == 3),
                colorbar=dict(tickformat=".0%",
                              tickfont=dict(family=FONT, color=GREY, size=10),
                              outlinecolor=DG, outlinewidth=1) if col_idx == 3 else None,
                hoverongaps=False,
            ),
            row=1, col=col_idx,
        )

    fig.update_layout(
        title=dict(text=f"{ticker}  ·  {name}  -  CPI vs Unemployment vs Combined",
                   font=dict(family=FONT, size=19, color="#111111"),
                   x=0.03, xanchor="left"),
        paper_bgcolor=BG, plot_bgcolor=PANEL,
        font=dict(family=FONT, color="#111111", size=13),
        height=400, width=1200,
        margin=dict(l=65, r=40, t=85, b=40),
        hoverlabel=dict(bgcolor="#ffffff", bordercolor=DG,
                        font=dict(family=FONT, color="#111111", size=12)),
    )
    for i in range(1, 4):
        fig.update_xaxes(side="top", tickfont=dict(family=FONT, color="#111111", size=11),
                         gridcolor=DG, linecolor=DG, row=1, col=i)
        fig.update_yaxes(autorange="reversed",
                         tickfont=dict(family=FONT, color="#111111", size=11),
                         gridcolor=DG, linecolor=DG, row=1, col=i)
    fig.show()

In [16]:
# Side-by-side comparisons for all tickers
for ticker in TICKER_META:
    plot_comparison(ticker)

## Part 5: Walk-Forward Strategy (Combined CPI + Unemployment)

In [31]:
# Set Configs
LONG_AVG_RET  =  0.02
LONG_MED_RET  =  0.00
LONG_MIN_N    =  15

SHORT_AVG_RET = -0.02
SHORT_MED_RET =  0.00
SHORT_MIN_N   =  15

In [32]:
train = df[df["year"] <= 2022].copy()
test  = df[df["year"] == 2023].copy()

print(f"Training set : {len(train):,} rows | years {sorted(train['year'].unique())}")
print(f"Test set     : {len(test):,}  rows | 2023 only")

Training set : 7,811 rows | years [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Test set     : 1,975  rows | 2023 only


In [33]:
# Identify signals on training data
combo_stats = (train.groupby(["ticker","entry_offset","exit_offset"])
                    .agg(
                        n        = ("pnl_pct","count"),
                        avg_ret  = ("pnl_pct","mean"),
                        med_ret  = ("pnl_pct","median"),
                        win_rate = ("pnl_pct", lambda x: (x>0).mean()),
                        std      = ("pnl_pct","std"),
                    )
                    .reset_index())

long_signals = combo_stats[
    (combo_stats["n"]       > LONG_MIN_N)  &
    (combo_stats["avg_ret"] > LONG_AVG_RET) &
    (combo_stats["med_ret"] > LONG_MED_RET)
].copy()
long_signals["side"] = "LONG"

short_signals = combo_stats[
    (combo_stats["n"]       > SHORT_MIN_N)   &
    (combo_stats["avg_ret"] < SHORT_AVG_RET) &
    (combo_stats["med_ret"] < SHORT_MED_RET)
].copy()
short_signals["side"] = "SHORT"

print(f"\nSignals identified from training period:")
print(f"  Long  signals : {len(long_signals)}")
print(f"  Short signals : {len(short_signals)}")

print(f"\nLong signals:")
print(long_signals[["ticker","entry_offset","exit_offset","n","avg_ret","med_ret","win_rate"]]
      .sort_values("avg_ret", ascending=False)
      .to_string(index=False, formatters={
          "avg_ret":  "{:.2%}".format,
          "med_ret":  "{:.2%}".format,
          "win_rate": "{:.0%}".format,
      }))

print(f"\nShort signals:")
print(short_signals[["ticker","entry_offset","exit_offset","n","avg_ret","med_ret","win_rate"]]
      .sort_values("avg_ret")
      .to_string(index=False, formatters={
          "avg_ret":  "{:.2%}".format,
          "med_ret":  "{:.2%}".format,
          "win_rate": "{:.0%}".format,
      }))


Signals identified from training period:
  Long  signals : 15
  Short signals : 50

Long signals:
ticker  entry_offset  exit_offset   n avg_ret med_ret win_rate
   XOP            -3            1  34  16.18%   7.01%      62%
   XLE            -2            1  20   8.82%   8.84%      75%
   XOP            -2            0  31   6.58%   1.33%      61%
   IWM            -2            1  49   5.89%   5.19%      55%
   XLE            -2            2  19   5.01%   1.52%      58%
   XLE            -2            0  21   4.85%   6.90%      62%
   XOP            -2            2  32   4.37%   3.49%      59%
   XLF            -3            0  39   4.23%   3.76%      56%
   XOP            -1            2  28   3.63%   3.87%      54%
   XOP             0            1  28   3.54%   1.28%      57%
   XOP            -3            2  28   3.21%   4.70%      61%
   XOP            -1            1  28   2.39%   2.56%      61%
   QQQ            -2            0 119   2.25%   2.49%      54%
   XOP            -

In [34]:
# Apply signals to test set
long_keys  = set(zip(long_signals["ticker"],
                     long_signals["entry_offset"],
                     long_signals["exit_offset"]))
short_keys = set(zip(short_signals["ticker"],
                     short_signals["entry_offset"],
                     short_signals["exit_offset"]))

test = test.copy()
test["signal"] = None
test.loc[test.apply(lambda r: (r["ticker"],r["entry_offset"],r["exit_offset"])
                    in long_keys,  axis=1), "signal"] = "LONG"
test.loc[test.apply(lambda r: (r["ticker"],r["entry_offset"],r["exit_offset"])
                    in short_keys, axis=1), "signal"] = "SHORT"

test_signals = test[test["signal"].notna()].copy()
test_signals["strategy_pnl"] = test_signals.apply(
    lambda r: r["pnl_pct"] if r["signal"]=="LONG" else -r["pnl_pct"], axis=1
)

print(f"\nTest trades: {len(test_signals)}")
print(f"  Long  trades : {(test_signals['signal']=='LONG').sum()}")
print(f"  Short trades : {(test_signals['signal']=='SHORT').sum()}")


Test trades: 983
  Long  trades : 118
  Short trades : 865


In [35]:
# Performance per ticker
print(f"\nPerformance by Ticker:")
ticker_perf = (test_signals.groupby(["ticker","signal"])
               .agg(
                   n        = ("strategy_pnl","count"),
                   avg_ret  = ("strategy_pnl","mean"),
                   win_rate = ("strategy_pnl", lambda x: (x>0).mean()),
                   total    = ("strategy_pnl","sum"),
               )
               .round(4))
print(ticker_perf.to_string(formatters={
    "avg_ret":  "{:.2%}".format,
    "win_rate": "{:.0%}".format,
    "total":    "{:.2f}".format,
}))


Performance by Ticker:
                 n avg_ret win_rate  total
ticker signal                             
GLD    SHORT   278  -2.82%      64%  -7.83
IWM    LONG     20  11.11%      50%   2.22
       SHORT    72  -7.29%      58%  -5.25
QQQ    LONG     40   3.12%      38%   1.25
       SHORT   129  -7.07%      56%  -9.11
SPY    SHORT   240 -11.49%      41% -27.58
XLE    LONG     11  10.44%      82%   1.15
       SHORT    26  -0.02%      54%  -0.01
XLF    LONG     11   6.31%      45%   0.69
       SHORT    85  -9.39%      38%  -7.98
XLK    SHORT    25 -12.86%      48%  -3.21
XOP    LONG     36   1.97%      47%   0.71
       SHORT    10  11.97%      90%   1.20


In [36]:
# Training vs Test comparison
print(f"\nTraining vs Test — Signal Combos:")
print(f"{'Ticker':<6} {'E':>3} {'X':>3} {'Side':<6} "
      f"{'Train Avg':>10} {'Train WR':>9} {'Train N':>8} "
      f"{'Test Avg':>9} {'Test WR':>8} {'Test N':>7}")

all_signals = pd.concat([long_signals, short_signals])
for _, row in all_signals.sort_values(["side","ticker"]).iterrows():
    t, e, x, side = row["ticker"], int(row["entry_offset"]), int(row["exit_offset"]), row["side"]
    test_sub = test_signals[(test_signals["ticker"]==t) &
                             (test_signals["entry_offset"]==e) &
                             (test_signals["exit_offset"]==x)]
    if len(test_sub) == 0:
        test_avg, test_wr, test_n = float("nan"), float("nan"), 0
    else:
        pnl = test_sub["strategy_pnl"]
        test_avg = pnl.mean(); test_wr = (pnl>0).mean(); test_n = len(pnl)

    print(f"{t:<6} {e:>+3} {x:>+3} {side:<6} "
          f"{row['avg_ret']:>10.2%} {row['win_rate']:>9.0%} {int(row['n']):>8} "
          f"{test_avg:>9.2%} {test_wr:>8.0%} {test_n:>7}")


Training vs Test — Signal Combos:
Ticker   E   X Side    Train Avg  Train WR  Train N  Test Avg  Test WR  Test N
IWM     -2  +1 LONG        5.89%       55%       49    11.11%      50%      20
QQQ     -2  +0 LONG        2.25%       54%      119     5.16%      41%      22
QQQ     -2  +3 LONG        2.08%       52%      116     0.63%      33%      18
XLE     -2  +0 LONG        4.85%       62%       21    15.22%     100%       4
XLE     -2  +1 LONG        8.82%       75%       20     4.91%      75%       4
XLE     -2  +2 LONG        5.01%       58%       19    11.44%      67%       3
XLF     -3  +0 LONG        4.23%       56%       39     6.31%      45%      11
XOP     -3  +1 LONG       16.18%       62%       34    -6.91%      43%       7
XOP     -3  +2 LONG        3.21%       61%       28    -0.71%      50%       4
XOP     -2  +0 LONG        6.58%       61%       31    20.74%      60%       5
XOP     -2  +2 LONG        4.37%       59%       32    -1.95%      20%       5
XOP     -1  +0 LO

In [37]:
# Cumulative PnL chart
all_periods = df.copy()
all_periods["signal"] = None
all_periods.loc[all_periods.apply(
    lambda r: (r["ticker"],r["entry_offset"],r["exit_offset"]) in long_keys, axis=1),
    "signal"] = "LONG"
all_periods.loc[all_periods.apply(
    lambda r: (r["ticker"],r["entry_offset"],r["exit_offset"]) in short_keys, axis=1),
    "signal"] = "SHORT"

strat = all_periods[all_periods["signal"].notna()].copy()
strat["strategy_pnl"] = strat.apply(
    lambda r: r["pnl_pct"] if r["signal"]=="LONG" else -r["pnl_pct"], axis=1)
strat["period"] = strat["year"].apply(lambda y: "train" if y <= 2022 else "test")

CAPITAL_PER_TRADE = 1_000

strat["dollar_pnl"] = strat["strategy_pnl"] * CAPITAL_PER_TRADE
strat = strat.sort_values("exit_date").reset_index(drop=True)
strat["cum_dollar_pnl"]  = strat.groupby("period")["dollar_pnl"].cumsum()
strat["overall_cum_pnl"] = strat["dollar_pnl"].cumsum()
strat["total_invested"]  = (strat.index + 1) * CAPITAL_PER_TRADE
strat["roi"]             = strat["overall_cum_pnl"] / strat["total_invested"]
strat["peak"]            = strat["overall_cum_pnl"].cummax()
strat["drawdown"]        = strat["overall_cum_pnl"] - strat["peak"]

print(f"\nOverall Strategy Performance  (${CAPITAL_PER_TRADE:,} per trade)")
for period, grp in strat.groupby("period"):
    r   = grp["dollar_pnl"]
    inv = len(grp) * CAPITAL_PER_TRADE
    roi = r.sum() / inv
    print(f"\n  {period.upper()} ({grp['year'].min()}-{grp['year'].max()})")
    print(f"    Trades          : {len(r)}")
    print(f"    Capital deployed: ${inv:,.0f}")
    print(f"    Total PnL       : ${r.sum():,.2f}")
    print(f"    ROI             : {roi:.2%}")
    print(f"    Win rate        : {(r>0).mean():.1%}")
    print(f"    Avg per trade   : ${r.mean():,.2f}")
    print(f"    Best trade      : ${r.max():,.2f}")
    print(f"    Worst trade     : ${r.min():,.2f}")

r   = strat["dollar_pnl"]
inv = len(strat) * CAPITAL_PER_TRADE
roi = r.sum() / inv
print(f"\n  FULL PERIOD")
print(f"    Trades          : {len(r)}")
print(f"    Capital deployed: ${inv:,.0f}")
print(f"    Total PnL       : ${r.sum():,.2f}")
print(f"    ROI             : {roi:.2%}")
print(f"    Win rate        : {(r>0).mean():.1%}")
print(f"    Avg per trade   : ${r.mean():,.2f}")
print(f"    Max Drawdown    : ${strat['drawdown'].min():,.2f}")
print(f"    Best trade      : ${r.max():,.2f}")
print(f"    Worst trade     : ${r.min():,.2f}")

fig = go.Figure()

for period, col, pname in [("train", "#2E74B5", "Training"),
                            ("test",  GREEN,     "Test")]:
    grp = strat[strat["period"]==period]
    fig.add_trace(go.Scatter(
        x=grp["exit_date"], y=grp["cum_dollar_pnl"],
        mode="lines", name=pname,
        line=dict(color=col, width=2),
        hovertemplate="%{x|%b %Y}<br>Cumulative PnL: $%{y:,.0f}<extra></extra>",
    ))

fig.add_vline(x=pd.Timestamp("2023-01-01").timestamp()*1000,
              line_color=DG, line_dash="dash",
              annotation_text="Test begins",
              annotation_font=dict(family=FONT, size=11, color=GREY))
fig.add_hline(y=0, line_color=DG, line_dash="dot")

fig.update_layout(
    title=dict(text=f"Long/Short CPI+Unemployment Straddle  ·  Cumulative PnL  (${CAPITAL_PER_TRADE:,} per trade)",
               font=dict(family=FONT, size=19, color="#111111"),
               x=0.03, xanchor="left"),
    paper_bgcolor=BG, plot_bgcolor=PANEL,
    font=dict(family=FONT, color="#111111", size=13),
    height=460, margin=dict(l=80, r=40, t=70, b=60),
    legend=dict(bgcolor=BG, bordercolor=DG, borderwidth=1,
                font=dict(family=FONT, size=12)),
    hoverlabel=dict(bgcolor="#ffffff", bordercolor=DG,
                    font=dict(family=FONT, color="#111111", size=13)),
    xaxis=dict(gridcolor=DG, linecolor=DG,
               tickfont=dict(family=FONT, color="#333333")),
    yaxis=dict(title="Cumulative PnL ($)",
               tickprefix="$", tickformat=",.0f",
               gridcolor=DG, linecolor=DG, zerolinecolor=DG,
               tickfont=dict(family=FONT, color="#333333"),
               title_font=dict(family=FONT, color="#333333", size=12)),
)
fig.show()


Overall Strategy Performance  ($1,000 per trade)

  TEST (2023-2025)
    Trades          : 3126
    Capital deployed: $3,126,000
    Total PnL       : $-32,024.36
    ROI             : -1.02%
    Win rate        : 56.4%
    Avg per trade   : $-10.24
    Best trade      : $3,145.93
    Worst trade     : $-3,043.90

  TRAIN (2016-2022)
    Trades          : 4503
    Capital deployed: $4,503,000
    Total PnL       : $177,179.35
    ROI             : 3.93%
    Win rate        : 62.2%
    Avg per trade   : $39.35
    Best trade      : $1,902.10
    Worst trade     : $-1,917.21

  FULL PERIOD
    Trades          : 7629
    Capital deployed: $7,629,000
    Total PnL       : $145,154.99
    ROI             : 1.90%
    Win rate        : 59.9%
    Avg per trade   : $19.03
    Max Drawdown    : $-91,579.21
    Best trade      : $3,145.93
    Worst trade     : $-3,043.90
